# Distilled Tiny RNNs — rare-policy robustness study

**Purpose:** test whether the rare group's diagnostic true-policy KL loss survives stronger solo regularization and longer student training, alongside overall NLL/KL and both checkpoint-selection controls. The teacher was also weak on the diagnostic subset; compare teacher and student scores before interpreting a gap as loss caused by distillation. This run changes only the training budget, solo regularization grid, and population seeds from notebook 03, using the same audited runtime, selection rules, and metrics.

This is a **combined-protocol robustness replication**, not an isolated training-duration experiment or proof of convergence. Report notebook 03 and notebook 04 separately; do not pool their different protocols into a ten-seed result.

This remains a synthetic, conditional first-stage experiment with a parametric rare group. It does not establish recovery of people's reasoning or reproduce the published method.

Open this notebook in a fresh Colab session and select a **T4 GPU**. The first cell downloads a fixed public revision without a GitHub token and mounts your Google Drive for results. Run the cells in order.

The full study uses **1,000 people, 100 trials per person, and a 2% rare group**, with **500 teacher steps, 4,000 student steps, and evaluation every 10 steps**. Each phase has a soft **10-minute** limit; each study invocation has a soft **30-minute** budget. Setup, completed steps, saving, and evaluation can add time. These are stopping limits, not a runtime or cost forecast. If the study pauses while this session is still live, **rerun ONLY the fourth code cell with the same RUN_NAME; do NOT use Run all or restart the runtime to resume**. No repeated-seed summary is available until every planned seed finishes. Disconnect the runtime when finished.

**Recovery after a disconnect:** if Colab has cleared the variables or replaced the runtime, run the first three code cells again to restore the verified source, setup, and smoke check, then resume the fourth code cell with the same RUN_NAME and saved Google Drive folder. The fourth-cell-only rule applies while the same session remains live.


In [ ]:
"""Download one pinned public GitHub revision for a Colab launcher.

This standard-library module is embedded into the launchers. Each invocation
installs into a fresh runtime-local destination; saved runs belong elsewhere.
"""
from pathlib import Path, PurePosixPath
import io
import os
import re
import shutil
import stat
import tempfile
import urllib.error
import urllib.parse
import urllib.request
import zipfile


_ARCHIVE_LIMIT = 50 * 1024 * 1024
_EXTRACTED_LIMIT = 100 * 1024 * 1024


class _GitHubRedirectHandler(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        parsed = urllib.parse.urlsplit(newurl)
        if (parsed.scheme != "https" or parsed.hostname != "codeload.github.com"
                or parsed.username is not None or parsed.password is not None
                or parsed.port not in (None, 443)):
            raise RuntimeError("GitHub returned an unexpected download destination")
        return super().redirect_request(req, fp, code, msg, headers, newurl)


def _extract_project_archive(payload, destination):
    """Validate all archive paths before extracting a single GitHub root."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if os.path.lexists(destination):
        raise ValueError("Use a fresh empty destination path for the project")
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        entries = archive.infolist()
        if not entries or len(entries) > 10000:
            raise ValueError("Invalid project archive entry count")
        roots, seen, planned = set(), set(), []
        total_size = 0
        for entry in entries:
            name = entry.filename
            components = name.rstrip("/").split("/")
            mode = stat.S_IFMT(entry.external_attr >> 16)
            if (name.startswith("/") or "\\" in name or any(p in ("", ".", "..") for p in components)
                    or mode not in (0, stat.S_IFREG, stat.S_IFDIR) or entry.flag_bits & 1):
                raise ValueError("Unsafe project archive entry")
            roots.add(components[0])
            relative = PurePosixPath(*components[1:])
            normalized = str(relative)
            if normalized in seen:
                raise ValueError("Duplicate project archive entry")
            seen.add(normalized)
            if len(components) == 1:
                if not entry.is_dir():
                    raise ValueError("Project archive must contain a root directory")
                continue
            total_size += entry.file_size
            if total_size > _EXTRACTED_LIMIT:
                raise ValueError("Project archive is too large")
            planned.append((entry, relative))
        if len(roots) != 1 or not planned:
            raise ValueError("Project archive must have one root directory")
        files = {str(path) for entry, path in planned if not entry.is_dir()}
        for _, path in planned:
            if any(str(parent) in files for parent in path.parents):
                raise ValueError("Conflicting project archive paths")
        staging = Path(tempfile.mkdtemp(prefix=".colab-project-", dir=destination.parent))
        try:
            for entry, relative in planned:
                target = staging.joinpath(*relative.parts)
                if entry.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(entry) as source, target.open("xb") as output:
                        shutil.copyfileobj(source, output, length=1024 * 1024)
            # Caller selects an unused path. Recheck to avoid replacing a path
            # that appeared while downloading/extracting.
            if os.path.lexists(destination):
                raise ValueError("Project destination already exists")
            staging.rename(destination)
        finally:
            if staging.exists():
                shutil.rmtree(staging)
    return destination


def fetch_public_project(repo, commit, destination):
    """Fetch and safely extract a public ZIP pinned to a full Git commit."""
    if (not isinstance(repo, str)
            or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9-]{0,38}/[A-Za-z0-9_.-]{1,100}", repo)
            or repo.rsplit("/", 1)[-1] in {".", ".."}):
        raise ValueError("Repository must be an owner/name pair")
    if not isinstance(commit, str) or not re.fullmatch(r"[0-9a-fA-F]{40}", commit):
        raise ValueError("A full 40-character Git commit is required")
    if os.path.lexists(destination):
        raise ValueError("Use a fresh destination path for the project")
    request = urllib.request.Request(
        f"https://github.com/{repo}/archive/{commit}.zip",
        headers={"User-Agent": "distilled-rnns-colab"},
    )
    try:
        opener = urllib.request.build_opener(_GitHubRedirectHandler())
        with opener.open(request, timeout=60) as response:
            payload = response.read(_ARCHIVE_LIMIT + 1)
    except urllib.error.HTTPError as error:
        if error.code == 404:
            raise RuntimeError(
                "GitHub could not find the pinned revision. Check that the repository is public."
            ) from None
        raise RuntimeError(f"GitHub project download failed (HTTP {error.code}).") from None
    except (OSError, RuntimeError, ValueError):
        raise RuntimeError("GitHub project download failed. Check your connection and try again.") from None
    if len(payload) > _ARCHIVE_LIMIT:
        raise RuntimeError("The project download exceeds the size limit")
    try:
        return _extract_project_archive(payload, destination)
    except Exception:
        raise RuntimeError("The downloaded project archive could not be safely installed") from None


from pathlib import Path
import os, sys, hashlib, json, subprocess, tempfile
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

EXPECTED_SOURCE_SHA256 = "63c789f6b9ef1cb7c8d32bd726ecfbfcf2c44aa3ac48ed5da18c5096cb008210"
GITHUB_REPOSITORY = "WeaversMask/distilled-tiny-rnns-benchmark"
PINNED_COMMIT = "8fb2803f54829aa6ce1326affcc10b55d4a79e8c"
ROBUSTNESS_REVISION_READY = True
if IN_COLAB and not ROBUSTNESS_REVISION_READY:
    raise RuntimeError("This robustness launcher is awaiting its verified runtime release. Use the completed reviewed notebook.")
if IN_COLAB:
    destination = Path(tempfile.mkdtemp(prefix="distilled-source-")) / "project"
    PROJECT = fetch_public_project(GITHUB_REPOSITORY, PINNED_COMMIT, destination)
    drive.mount("/content/drive")
    DRIVE_PROJECT = Path("/content/drive/MyDrive/distilled-tiny-rnns")
    RUNS = DRIVE_PROJECT / "runs"
else:
    PROJECT = Path(os.environ.get("DISTILLED_PROJECT_ROOT", Path.cwd())).resolve()
    if not (PROJECT / "pyproject.toml").exists() and (PROJECT.parent / "pyproject.toml").exists():
        PROJECT = PROJECT.parent
    RUNS = Path(os.environ.get("DISTILLED_OUTPUT_ROOT", str(PROJECT / "outputs/notebook")))

digest = hashlib.sha256()
for path in sorted((PROJECT / "src/distilled_rnns").glob("*.py")):
    digest.update(path.name.encode())
    digest.update(path.read_bytes())
assert digest.hexdigest() == EXPECTED_SOURCE_SHA256, "Downloaded source differs from the verified notebook revision."
os.chdir(PROJECT)
RUNS.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT)
print("Saved runs:", RUNS)


## Install and verify

The source fingerprint is checked before training. Setup activates the downloaded package in this running kernel and rejects modules left over from another revision. The test suite runs before the smoke test, including GPU-specific checks when CUDA is present. Exact package versions are recorded with every run.


In [ ]:
# A different notebook revision may already be imported in this kernel.
# Never mix its modules with this verified source download.
expected_package = (PROJECT / "src/distilled_rnns").resolve()
for module_name, module in list(sys.modules.items()):
    if module_name == "distilled_rnns" or module_name.startswith("distilled_rnns."):
        module_file = getattr(module, "__file__", None)
        if module_file is None or not Path(module_file).resolve().is_relative_to(expected_package):
            raise RuntimeError("Another project revision is loaded. Restart the Colab session, then run this notebook from its first cell.")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", ".[test]"], check=True)
# Editable installs create a startup path file; this kernel is already running.
# Activate the verified source immediately in both Colab and local kernels.
import importlib
source_dir = str((PROJECT / "src").resolve())
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
from distilled_rnns.config import ExperimentConfig
print("Project import: ready")
import torch
print("PyTorch:", torch.__version__)
if IN_COLAB and not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing; this notebook will not silently run the Colab pilot on CPU.")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU (local verification only)")
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], check=True)


## Small robustness smoke test

This tiny study checks the separate tuning split, both checkpoint-selection methods, fixed references, repeated seeds, and resume. Its scores are software checks, not scientific findings.


In [ ]:
from distilled_rnns.baseline_audit import BaselineAuditConfig, run_baseline_audit

SMOKE_RUN = RUNS / ("robustness-smoke-" + EXPECTED_SOURCE_SHA256[:12])
smoke_config = BaselineAuditConfig.from_file(PROJECT / "configs/robustness_smoke.json")
smoke = run_baseline_audit(smoke_config, SMOKE_RUN, device=DEVICE)
assert smoke["status"] == "complete", smoke
smoke_resumed = run_baseline_audit(smoke_config, SMOKE_RUN, device=DEVICE)
assert smoke_resumed["groups"] == smoke["groups"], "Resumed results differ"
for field in ("solo_l2", "solo_selection", "distilled_selection"):
    assert smoke_resumed["calibration"][field] == smoke["calibration"][field]
print("Robustness smoke test and resume passed.")


## Run the bounded robustness study

**Calibration uses a fresh population, seed 20261301.** For each person, 60% of trials train the models, 20% select saved checkpoints, and a separate final 20% choose settings. That last calibration split is **tuning data**, not an independent test set. No calibration test result is claimed.

Solo candidates retain L2 strengths **0, 0.001, 0.01, and 0.1** and add **0.3, 1.0, and 3.0**. Each fitted trajectory supplies two alternatives: **individual selection** chooses each person's best checkpoint; **population selection** chooses one step using mean validation NLL across the population, while every person's model weights remain independent. Tracking both alternatives does not train the same trajectory twice. The outer tuning split chooses the solo strength and selection method, and separately chooses the distilled selection method. Exact ties favor smaller L2, then population selection. Settings are selected over the population, dominated by the 98% typical majority; neither rare labels nor true-policy scores choose them.

**Evaluation freezes those choices** before testing five fresh populations: **20261311, 20261321, 20261331, 20261341, 20261351**. These are integer random seeds, not dates. Each evaluation population keeps its own 60% training, 20% checkpoint-selection, and 20% test split. Test scores do not choose settings.

The report includes the teacher, calibrated solo and distilled pipelines, both matched checkpoint-selection comparisons, a **fixed solo L2=0.01 control with individual selection and 4,000 training steps**, a fixed **50/50 reference**, and a **choice-frequency reference** estimated only from training choices with Laplace smoothing. The fixed solo control is trained longer here than in notebook 03; it is not the exact old recipe. The matched comparisons hold the checkpoint rule fixed; the training objective and solo L2 still differ.

Keep the default protocol for this run. Changing source or settings requires a new run name. Do not use these test scores to choose settings and then describe the same seeds as an independent assessment.


In [ ]:
RUN_NAME = "robustness-v1"  # Keep for resume; change for new settings or source.
CONFIG_NAME = "robustness" if IN_COLAB else os.environ.get("DISTILLED_ROBUSTNESS_CONFIG", "robustness_smoke")
config = BaselineAuditConfig.from_file(PROJECT / "configs" / f"{CONFIG_NAME}.json")
print(json.dumps(config.to_dict(), indent=2))
report = run_baseline_audit(config, RUNS / RUN_NAME, device=DEVICE)
if report["status"] != "complete":
    print(report["message"])
    print("Resume: rerun ONLY the fourth code cell with the same RUN_NAME. Do NOT use Run all or restart the runtime. No aggregation until all planned seeds complete.")


## Results and what to inspect

Prediction error (NLL) and policy mismatch (KL) are lower when better. A constant 50/50 prediction has NLL **0.6931**. The population-level means and ranges below describe the five fresh seeds; they are not significance tests. Calibration is excluded.

The main comparison uses the independently calibrated solo and distilled pipelines. The matched-method rows hold the checkpoint rule fixed; the training objective and solo L2 still differ, so these rows do not isolate a pure causal effect of distillation. Selected-pipeline rows can duplicate their matching individual/population rows.

The selection-audit figures compare scores on checkpoint-selection targets with separate calibration tuning targets or fresh-population test targets. They help show how well the selected checkpoints generalize beyond the data used to choose them. Calibration tuning scores remain part of model selection, not an independent test result.

Diagnostic scores use the same predeclared subset and report its support. Missing support remains unavailable. **Diagnostic KL loss is distilled minus the stated comparison model**; a positive value indicates worse true-policy fidelity for the distilled model. Both paired tables show this quantity. The per-seed table also compares the teacher and calibrated students on that same subset, because an already weak teacher can limit the interpretation. These descriptive comparisons do not establish causality or statistical significance.

The inherited plots label the fixed solo control **"Previous solo recipe"**. Here that label means L2=0.01 with individual checkpoint selection and this run's student training budget: **4,000 steps in the full study**, not the original 1,000-step recipe.

Inspect the learning curves and training-boundary warnings before claiming convergence. These models predict choices conditional on observed histories; they do not establish recovery of the generating mechanism on new free-running histories. If the run is incomplete and the same session remains live, rerun ONLY the fourth code cell with the same RUN_NAME; do NOT use Run all or restart the runtime to resume. If variables were cleared after a disconnect, follow the recovery instructions above first. Run this results cell again after the study finishes.


In [ ]:
if report["status"] == "complete":
    from IPython.display import display, Image, Markdown
    modes = {"individual": "one checkpoint per person", "population": "one checkpoint for the population"}
    calibration = report["calibration"]
    print("Frozen solo L2 strength:", calibration["solo_l2"])
    print("Solo selection:", modes.get(calibration["solo_selection"], calibration["solo_selection"]))
    print("Distilled selection:", modes.get(calibration["distilled_selection"], calibration["distilled_selection"]))
    print("These settings were selected on separate calibration tuning targets, before fresh-seed test scores.")
    for warning in report.get("convergence_warnings", []):
        print("CHECK REQUIRED:", warning)

    def _score_text(summary, paired=False):
        support = f"({summary['supported_seed_count']}/{summary['planned_seed_count']} seeds)"
        if summary["mean"] is None:
            return "unavailable " + support
        if paired:
            return f"{summary['mean']:+.4f} [{summary['min']:+.4f}, {summary['max']:+.4f}] {support}"
        return f"{summary['mean']:.4f} {support}"

    labels = {"teacher": "Teacher", "solo": "Calibrated solo", "distilled": "Calibrated distilled",
              "coinflip": "Fixed 50/50", "frequency": "Training choice frequency",
              "solo_legacy": f"Fixed solo L2=0.01, individual ({config.base.student_steps:,} steps)",
              "solo_individual": "Solo: individual checkpoint", "solo_population": "Solo: population checkpoint",
              "distilled_individual": "Distilled: individual checkpoint", "distilled_population": "Distilled: population checkpoint"}
    display(Markdown("**Scores:** seed means in nats per trial; lower is better. Parentheses show supported/planned seeds. "
                     "**Paired differences:** mean [minimum, maximum]. Positive NLL gain favors distillation; positive KL loss "
                     "means worse policy fidelity. The fixed 50/50 NLL reference is 0.6931. These are descriptive summaries."))
    for group_name, group in report["groups"].items():
        rows = [f"**{group_name.title()} group**", "", "| Model | NLL | KL | Diagnostic NLL | Diagnostic KL |",
                "| --- | ---: | ---: | ---: | ---: |"]
        for name in labels:
            metrics = group["models"][name]
            values = [_score_text(metrics[metric]) for metric in ("nll", "kl", "diagnostic_nll", "diagnostic_kl")]
            rows.append("| " + " | ".join([labels[name], *values]) + " |")
        rows.extend(["", "| Calibrated distilled versus | NLL gain | KL loss | Diagnostic NLL gain | Diagnostic KL loss |",
                     "| --- | ---: | ---: | ---: | ---: |"])
        for baseline in ("solo", "solo_legacy", "coinflip", "frequency"):
            metrics = group["comparisons"][baseline]
            values = [_score_text(metrics[metric], paired=True)
                      for metric in ("likelihood_gain", "fidelity_loss", "diagnostic_likelihood_gain", "diagnostic_fidelity_loss")]
            rows.append("| " + " | ".join([labels[baseline], *values]) + " |")
        rows.extend(["", "| Matched method: distilled versus solo | NLL gain | KL loss | Diagnostic NLL gain | Diagnostic KL loss |",
                     "| --- | ---: | ---: | ---: | ---: |"])
        for mode in ("individual", "population"):
            metrics = group["matched_comparisons"][mode]
            values = [_score_text(metrics[metric], paired=True)
                      for metric in ("likelihood_gain", "fidelity_loss", "diagnostic_likelihood_gain", "diagnostic_fidelity_loss")]
            rows.append("| " + " | ".join([modes[mode], *values]) + " |")
        support = [f"{row['seed']}: {row['diagnostic_count']} trials / {row['diagnostic_subject_count']} people"
                   for row in group["per_seed"]]
        rows.extend(["", "Diagnostic test support — " + "; ".join(support) + "."])
        rows.extend(["", "**Diagnostic true-policy KL by seed** (lower is better; same diagnostic subset)", "",
                     "| Seed | Teacher | Calibrated solo | Calibrated distilled | Distilled minus solo |",
                     "| --- | ---: | ---: | ---: | ---: |"])
        diagnostic_differences = []
        for row in group["per_seed"]:
            values = [row["models"][name]["diagnostic_kl"] for name in ("teacher", "solo", "distilled")]
            scores = ["unavailable" if value is None else f"{value:.4f}" for value in values]
            difference = row["comparisons"]["solo"]["diagnostic_fidelity_loss"]
            difference_text = "unavailable" if difference is None else f"{difference:+.4f}"
            if difference is not None:
                diagnostic_differences.append(difference)
            rows.append("| " + " | ".join([str(row["seed"]), *scores, difference_text]) + " |")
        higher = sum(value > 0 for value in diagnostic_differences)
        lower = sum(value < 0 for value in diagnostic_differences)
        tied = sum(value == 0 for value in diagnostic_differences)
        primary_diagnostic = group["comparisons"]["solo"]["diagnostic_fidelity_loss"]
        rows.extend(["", "Calibrated pipeline diagnostic KL difference, mean [minimum, maximum]: "
                     + _score_text(primary_diagnostic, paired=True) + ".",
                     f"Direction counts: {higher} higher, {lower} lower, {tied} tied; "
                     f"{len(diagnostic_differences)}/{primary_diagnostic['planned_seed_count']} seeds supported. "
                     "Higher means worse true-policy fidelity for distilled students; unavailable seeds are excluded."])
        display(Markdown("\n".join(rows)))
    timing = report["timing"]
    print(f"Recorded training time: {timing['cumulative_training_seconds'] / 60:.2f} minutes.")
    units = timing["t4_equivalent_units_at_user_rate"]
    if units is None:
        print("T4 compute-unit proxy unavailable for this device history.")
    else:
        print(f"Training-only T4 compute-unit proxy: {units:.4f} units at your supplied rate.")
    print("Includes all calibration and evaluation fits; setup, evaluation and idle billing are additional.")
    output_dir = Path(report["output_dir"])
    print(f'Plot label note: "Previous solo recipe" means fixed solo L2=0.01 with individual checkpoint selection and {config.base.student_steps:,} training steps in this run, not the original 1,000-step recipe.')
    selection_plots = sorted(output_dir.rglob("selection_audit.png"))
    calibration_plot = output_dir / "calibration" / "selection_audit.png"
    replication_plots = [plot for plot in selection_plots if plot.parent.name.startswith("seed-")]
    shown_selection = ([calibration_plot] if calibration_plot.is_file() else []) + replication_plots[:1]
    for plot in shown_selection:
        print("Checkpoint-selection audit:", plot.parent.name)
        display(Image(filename=str(plot)))
    for plot in selection_plots:
        if plot not in shown_selection:
            print("Additional selection-audit figure:", plot)
    convergence_plots = sorted(output_dir.rglob("convergence.png"))
    replication_curves = [plot for plot in convergence_plots if plot.parent.name.startswith("seed-")]
    shown_convergence = (replication_curves or convergence_plots)[:1]
    for plot in shown_convergence:
        print("Training curves:", plot.parent.name)
        display(Image(filename=str(plot)))
    for plot in convergence_plots:
        if plot not in shown_convergence:
            print("Additional convergence figure:", plot)
    print("Saved:", output_dir)
    print("Keep the entire robustness-v1 folder, including protocol.json, summary.json, calibration, every seed folder, checkpoints, predictions and environment records.")
else:
    print("Robustness study incomplete. Rerun ONLY the fourth code cell with the same RUN_NAME. Do NOT use Run all or restart the runtime. No aggregation until all planned seeds complete.")
